In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.isotonic import IsotonicRegression           
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import shap

# ---------- 0. 路径 ----------
DATA_FILE   = "Belonging_scored.xlsx"
UNSCORED    = "images.xlsx"
OUT_DIR     = Path("D:\Desktop\output_Belonging")
OUT_DIR.mkdir(exist_ok=True)

PREDICT_XLSX = OUT_DIR / "images_predicted_Belonging.xlsx"
BSWARM_PNG   = OUT_DIR / "shap_beeswarm_Belonging.png"
BAR_PNG      = OUT_DIR / "shap_bar_Belonging.png"

# ---------- 1. 常量 ----------
TARGET = "Belonging"
FEATS  = ["Road","Building","Pole Group","Indicator","Vegetation","Sky",
          "Person","Car","Motorcycle","Bicycle","Clothes","Trash Can",
          "Riverway","Signboard","Air Conditioner Condenser","Festival Elements"]

# ---------- 2. 读取已打分数据 ----------
df = pd.read_excel(DATA_FILE)
df[TARGET] = df[TARGET].round(5)           

X = df[FEATS].copy()
y = df[TARGET].values

rng = np.random.default_rng(42)
for col in FEATS:
    zero_mask = X[col] == 0
    if zero_mask.any():
        X.loc[zero_mask, col] += rng.uniform(0.01, 0.03, size=zero_mask.sum())
# ————————————————————————————————————————————————

# ---------- 3. 划分 & 训练 ----------
X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.20, random_state=42)

rf = RandomForestRegressor(
        n_estimators     = 400,
        max_depth        = 9,
        min_samples_leaf = 5,        
        max_features     = "sqrt",
        random_state     = 42,
        n_jobs           = -1
     )
rf.fit(X_tr, y_tr)

# ---------- 4. 单调校准----------
iso = IsotonicRegression(
        y_min=y.min(),               
        y_max=y.max(),
        increasing=True,
        out_of_bounds="clip" 
     )
iso.fit(rf.predict(X), y)             

# ---------- 5. 评估 ----------
def _metrics(t, p): return r2_score(t, p), mean_squared_error(t, p, squared=False)

raw_tr,  raw_te  = rf.predict(X_tr), rf.predict(X_te)
cal_tr,  cal_te  = iso.transform(raw_tr), iso.transform(raw_te)

r2_tr,  rmse_tr  = _metrics(y_tr,  cal_tr)  
r2_te,  rmse_te  = _metrics(y_te,  cal_te)
r2_all, rmse_all = _metrics(y,     iso.transform(rf.predict(X)))


print(f"Train   R² = {r2_tr :.4f} | RMSE = {rmse_tr :.4f}")
print(f"Test    R² = {r2_te :.4f} | RMSE = {rmse_te :.4f}")
print(f"Overall R² = {r2_all:.4f} | RMSE = {rmse_all:.4f}")


# ---------- 6. SHAP  ----------
explainer   = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X)

plt.figure(figsize=(7,6), dpi=300)
shap.summary_plot(shap_values, X, feature_names=FEATS, show=False)
plt.title("SHAP Summary (Beeswarm) – Belonging")
plt.savefig(BSWARM_PNG, bbox_inches="tight", dpi=300)
plt.close()

plt.figure(figsize=(7,6), dpi=300)
shap.summary_plot(shap_values, X, feature_names=FEATS,
                  plot_type="bar", show=False)
plt.title("Feature Importance (mean |SHAP value|)")
plt.savefig(BAR_PNG, bbox_inches="tight", dpi=300)
plt.close()

# ---------- 7. 预测未打分样本 ----------
df_new = pd.read_excel(UNSCORED)
X_new  = df_new[FEATS].copy()


for col in FEATS:
    m0 = X_new[col] == 0
    if m0.any():
        X_new.loc[m0, col] += rng.uniform(0.01, 0.03, size=m0.sum())

raw_pred   = rf.predict(X_new)
calib_pred = iso.transform(raw_pred).round(5)  
df_new[TARGET] = calib_pred

df_new.to_excel(PREDICT_XLSX, index=False, float_format="%.5f")

print("✔ 预测结果:", PREDICT_XLSX)
print("✔ SHAP 图:", BSWARM_PNG, BAR_PNG)


Train   R² = 0.9499 | RMSE = 0.1110
Test    R² = 0.9123 | RMSE = 0.1563
Overall R² = 0.9417 | RMSE = 0.1214
✔ 预测结果: D:\Desktop\output_Belonging\images_predicted_Belonging.xlsx
✔ SHAP 图: D:\Desktop\output_Belonging\shap_beeswarm_Belonging.png D:\Desktop\output_Belonging\shap_bar_Belonging.png
